# 🛡️ MONAI Aegis — De-identification Pipeline

This notebook walks you through the complete Aegis pipeline for de-identifying medical images (DICOM, JPEG, PNG).

**Pipeline Architecture (3 Zones):**
1. **Ingestion Zone** — `LoadDicomRawd` reads files once, caches `pydicom.Dataset` in memory, enriches `MetaTensor.meta`
2. **Logic Zone** — `RedactPixelPHId` (OCR + NER) and `ScrubDicomMetadatad` (metadata scrub) — purely in-memory
3. **Persistence Zone** — `SaveDicomd` writes de-identified files to disk

**MONAI API Compliance:** All transforms inherit from `MapTransform`, `InvertibleTransform`, or `ThreadUnsafe` as appropriate.

---
## 1. Setup & Imports

In [ ]:
import sys
import os

# Ensure the monai_aegis package is on the path
sys.path.insert(0, os.path.join(os.getcwd(), 'monai_aegis'))

print(f'Working directory: {os.getcwd()}')
print(f'Python: {sys.version}')

In [ ]:
# Verify all dependencies are installed
try:
    import torch
    import monai
    import pydicom
    import easyocr
    import yaml
    import numpy as np
    from PIL import Image
    print(f'✅ torch    {torch.__version__}')
    print(f'✅ monai    {monai.__version__}')
    print(f'✅ pydicom  {pydicom.__version__}')
    print(f'✅ easyocr  {easyocr.__version__}')
    print(f'✅ All dependencies OK')
except ImportError as e:
    print(f'❌ Missing dependency: {e}')
    print('Run: pip install -e monai_aegis/')

---
## 2. Load Configuration

The pipeline is driven by `config.yaml`, which controls OCR settings, NER model, clinical allowlists, PHI heuristics, and PII tag actions.

In [ ]:
import yaml

CONFIG_PATH = 'monai_aegis/config/config.yaml'

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

print('=== OCR Settings ===')
for k, v in config['ocr'].items():
    print(f'  {k}: {v}')

print(f'\n=== NER Settings ===')
print(f'  enabled: {config["ner"]["enabled"]}')
print(f'  model:   {config["ner"]["model_name"]}')
print(f'  device:  {config["ner"]["device"]}')
print(f'  PHI labels ({len(config["ner"]["phi_labels"])}): {config["ner"]["phi_labels"][:5]}...')

print(f'\n=== PII Mapping ===')
for tag, action in config['pii_mapping'].items():
    print(f'  {tag} → {action}')

---
## 3. Inspect Input Files

Let's see what files are available in `staging_input/`.

In [ ]:
INPUT_DIR = 'staging_input'
OUTPUT_DIR = 'staging_output'

files = sorted([f for f in os.listdir(INPUT_DIR) 
                if f.lower().endswith(('.dcm', '.jpg', '.jpeg', '.png'))])

print(f'Found {len(files)} input files:\n')
for f in files:
    size_kb = os.path.getsize(os.path.join(INPUT_DIR, f)) / 1024
    ext = os.path.splitext(f)[1].upper()
    print(f'  {ext:6s}  {size_kb:7.1f} KB  {f}')

---
## 4. Preview an Input Image (Before De-identification)

View the original image with PHI visible.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import pydicom
import numpy as np

def show_image(filepath, title=''):
    """Display a DICOM or standard image file."""
    if filepath.lower().endswith('.dcm'):
        ds = pydicom.dcmread(filepath)
        img = ds.pixel_array
    else:
        img = np.array(Image.open(filepath))
    
    plt.figure(figsize=(10, 8))
    if img.ndim == 2:
        plt.imshow(img, cmap='gray')
    else:
        plt.imshow(img)
    plt.title(title, fontsize=14)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# ⬇️ Change this to any file from staging_input/
SAMPLE_FILE = '202601170918250004ABD.JPG'

show_image(os.path.join(INPUT_DIR, SAMPLE_FILE), 
           title=f'BEFORE — {SAMPLE_FILE} (PHI visible)')

---
## 5. Build & Run the Pipeline

Build the full 4-step pipeline and process a single file.

In [ ]:
from transforms.pipeline import build_pipeline

# Build the pipeline (loads config, initializes all transforms)
pipeline = build_pipeline(
    config_path=os.path.abspath(CONFIG_PATH),
    output_dir=OUTPUT_DIR
)

print('✅ Pipeline built successfully')
print(f'   Steps: LoadDicomRawd → RedactPixelPHId → ScrubDicomMetadatad → SaveDicomd')

In [ ]:
# Process one file
sample_path = os.path.join(INPUT_DIR, SAMPLE_FILE)
result = pipeline({'image': sample_path})

print(f'✅ Processed: {SAMPLE_FILE}')
print(f'\nResult dictionary keys: {list(result.keys())}')

---
## 6. Inspect Redaction Statistics

The pipeline attaches redaction statistics to the result dictionary.

In [ ]:
stats = result.get('image_redaction_stats', {})

print('=== Redaction Statistics ===')
print(f'  Total text detections:   {stats.get("total_detections", 0)}')
print(f'  Redacted (PHI):          {stats.get("redacted_count", 0)}')
print(f'  Preserved (clinical):    {stats.get("preserved_count", 0)}')
print(f'  Low confidence (skipped): {stats.get("low_confidence_count", 0)}')

if stats.get('low_confidence_count', 0) > 0:
    print('\n⚠️  This image has low-confidence regions — should be flagged for manual review.')
else:
    print('\n✅  All text regions processed with confidence above threshold.')

---
## 7. View the De-identified Output (After)

Compare the original and de-identified images side by side.

In [ ]:
# For standard images, extract from the result tensor
img_tensor = result['image']

if hasattr(img_tensor, 'cpu'):
    img_array = img_tensor.cpu().numpy()
else:
    img_array = np.array(img_tensor)

# Convert channel-first (C, H, W) → (H, W, C)
if img_array.ndim == 3:
    if img_array.shape[0] == 1:
        img_array = img_array.squeeze(0)
    elif img_array.shape[0] == 3:
        img_array = np.moveaxis(img_array, 0, -1)

# Normalize float to uint8
if img_array.dtype in [np.float32, np.float64]:
    if img_array.max() <= 1.1:
        img_array = (img_array * 255).astype(np.uint8)
    else:
        img_array = img_array.astype(np.uint8)

# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

original = np.array(Image.open(sample_path))
axes[0].imshow(original)
axes[0].set_title(f'BEFORE — {SAMPLE_FILE}', fontsize=13, color='red')
axes[0].axis('off')

if img_array.ndim == 2:
    axes[1].imshow(img_array, cmap='gray')
else:
    axes[1].imshow(img_array)
axes[1].set_title(f'AFTER — PHI Redacted', fontsize=13, color='green')
axes[1].axis('off')

plt.suptitle('MONAI Aegis De-identification', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Batch Process All Files

Process every file in `staging_input/` and report results.

In [ ]:
import shutil

NOT_PROCESSED_DIR = 'staging_not_processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(NOT_PROCESSED_DIR, exist_ok=True)

processed, flagged, errors = 0, 0, 0

for filename in files:
    filepath = os.path.join(INPUT_DIR, filename)
    try:
        result = pipeline({'image': filepath})
        stats = result.get('image_redaction_stats', {})
        low_conf = stats.get('low_confidence_count', 0)
        
        if low_conf > 0:
            shutil.copy2(filepath, os.path.join(NOT_PROCESSED_DIR, filename))
            print(f'⚠️  {filename} → staging_not_processed/ (low confidence)')
            flagged += 1
            continue
        
        # Save non-DICOM images manually
        if not filename.lower().endswith('.dcm'):
            img_out = result['image']
            if hasattr(img_out, 'cpu'):
                img_out = img_out.cpu().numpy()
            else:
                img_out = np.array(img_out)
            if img_out.ndim == 3:
                if img_out.shape[0] == 1:
                    img_out = img_out.squeeze(0)
                elif img_out.shape[0] == 3:
                    img_out = np.moveaxis(img_out, 0, -1)
            if img_out.dtype in [np.float32, np.float64]:
                img_out = (img_out * 255).astype(np.uint8) if img_out.max() <= 1.1 else img_out.astype(np.uint8)
            Image.fromarray(img_out).save(os.path.join(OUTPUT_DIR, filename))
        
        print(f'✅  {filename} → staging_output/')
        processed += 1
        
    except Exception as e:
        print(f'❌  {filename} — Error: {e}')
        errors += 1

print(f'\n{"=" * 50}')
print(f'  Processed:      {processed}')
print(f'  Flagged:        {flagged}')
print(f'  Errors:         {errors}')
print(f'{"=" * 50}')

---
## 9. Inspect a DICOM File (Before & After Metadata Scrubbing)

Compare DICOM tags before and after scrubbing.

In [ ]:
DICOM_FILE = 'U0000001.dcm'  # ⬇️ Change to any .dcm in staging_input/

# Tags we care about (from pii_mapping in config)
pii_tags = {
    '(0010,0010)': 'PatientName',
    '(0010,0020)': 'PatientID',
    '(0008,0020)': 'StudyDate',
    '(0008,0050)': 'AccessionNumber',
    '(0008,0090)': 'ReferringPhysicianName',
    '(0008,0080)': 'InstitutionName',
}

def get_tag_value(ds, tag_str):
    """Safely get a DICOM tag value."""
    clean = tag_str.strip('() ').replace(' ', '')
    parts = clean.split(',')
    tag = pydicom.tag.Tag(int(parts[0], 16), int(parts[1], 16))
    if tag in ds:
        return str(ds[tag].value)
    return '<REMOVED>'

# Load original
ds_original = pydicom.dcmread(os.path.join(INPUT_DIR, DICOM_FILE))

# Load scrubbed (if available)
scrubbed_path = os.path.join(OUTPUT_DIR, DICOM_FILE)
if os.path.exists(scrubbed_path):
    ds_scrubbed = pydicom.dcmread(scrubbed_path)
    
    print(f'DICOM Tag Comparison: {DICOM_FILE}')
    print(f'{"Tag":<18} {"Field":<26} {"BEFORE":<30} {"AFTER"}')
    print('-' * 100)
    for tag, name in pii_tags.items():
        before = get_tag_value(ds_original, tag)
        after = get_tag_value(ds_scrubbed, tag)
        changed = '🔴' if before != after else '⚪'
        print(f'{changed} {tag:<16} {name:<26} {before:<30} {after}')
else:
    print(f'⚠️  Scrubbed DICOM not found at {scrubbed_path}')
    print(f'   Run the batch processing cell above first.')

---
## 10. Using Individual Transforms

You can use each transform independently for fine-grained control.

In [ ]:
from transforms.io import LoadDicomRawd
from transforms.pixel import RedactPixelPHId
from transforms.metadata import ScrubDicomMetadatad
from monai.transforms import MapTransform, InvertibleTransform

# Step 1: Load (Ingestion Zone)
loader = LoadDicomRawd(keys=['image'])
data = loader({'image': os.path.join(INPUT_DIR, SAMPLE_FILE)})

print(f'After LoadDicomRawd:')
print(f'  Tensor shape:       {data["image"].shape}')
print(f'  Tensor dtype:       {data["image"].dtype}')

# Enriched MetaTensor metadata
meta = data['image'].meta
print(f'\n=== Enriched MetaTensor.meta ===')
for k, v in meta.items():
    print(f'  {k}: {v}')

# Verify meta_dict is a live reference (not a copy)
is_ref = data['image_meta_dict'] is data['image'].meta
print(f'\n  meta_dict is live reference: {"✅ Yes" if is_ref else "❌ No (detached copy)"}')

if 'image_dicom_dataset' in data:
    print(f'  Cached dataset:     ✅ pydicom.Dataset in memory')
else:
    print(f'  Cached dataset:     — (non-DICOM file)')

In [ ]:
# Step 2: Redact (Logic Zone)
redactor = RedactPixelPHId(keys=['image'], config=config)
data = redactor(data)

stats = data.get('image_redaction_stats', {})
print(f'After RedactPixelPHId:')
print(f'  Total detections:  {stats.get("total_detections", 0)}')
print(f'  Redacted:          {stats.get("redacted_count", 0)}')
print(f'  Preserved:         {stats.get("preserved_count", 0)}')

---
## 11. MONAI API Compliance Verification

Verify that all transforms inherit from the correct MONAI base classes.

In [ ]:
from transforms.io import LoadDicomRawd, SaveDicomd
from transforms.pixel import RedactPixelPHId
from transforms.metadata import ScrubDicomMetadatad
from monai.transforms import MapTransform, InvertibleTransform, Transform, ThreadUnsafe

compliance = [
    ('LoadDicomRawd',         LoadDicomRawd,         [MapTransform]),
    ('RedactPixelPHId',       RedactPixelPHId,       [MapTransform, InvertibleTransform]),
    ('ScrubDicomMetadatad',   ScrubDicomMetadatad,   [MapTransform]),
    ('SaveDicomd',            SaveDicomd,            [MapTransform, ThreadUnsafe]),
]

print(f'{"Transform":<24} {"Expected Bases":<40} {"Status"}')
print('=' * 80)
for name, cls, bases in compliance:
    bases_str = ', '.join(b.__name__ for b in bases)
    status = '✅ PASS' if all(issubclass(cls, b) for b in bases) else '❌ FAIL'
    print(f'{name:<24} {bases_str:<40} {status}')

---
## 12. Run Unit Tests

Run the full test suite (34 tests) from within the notebook.

In [ ]:
!PYTHONPATH=monai_aegis python -m unittest discover tests/unit -v

---
## 13. Configuration Quick-Reference

| Setting | Location | Effect |
|---------|----------|--------|
| `ocr.confidence_threshold` | config.yaml | Lower = more text detected; Higher = stricter |
| `ner.enabled` | config.yaml | `true` = Stanford NER; `false` = regex safelist |
| `ner.device` | config.yaml | `cpu`, `cuda`, or `mps` |
| `ner.phi_labels` | config.yaml | NER entity types treated as PHI |
| `ner.clinical_allowlist` | config.yaml | Terms that are NEVER redacted |
| `ner.clinical_patterns` | config.yaml | Regex for clinical text (preserved) |
| `ner.phi_heuristic_patterns` | config.yaml | Regex for PHI in short fragments (redacted) |
| `pii_mapping` | config.yaml | DICOM tag actions: REMOVE / ZERO / DUMMY |